In [1]:
from pyspark.sql import SparkSession
import sys
sys.path.append('/home/jovyan/work/src') 

from csv_to_parquet import convert_csv_to_parquet_spark


In [2]:
spark = (
    SparkSession.builder
    .appName("CsvToParquetPipeline")
    .master("local[*]")  # todos los cores disponibles en el contenedor
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

In [3]:
# Nombre del CSV dentro de /data en el contenedor
input_csv = ["BEHAVIOURAL.csv", "CLIENTS.csv"]
output_parquet = ["BEHAVIOURAL", "CLIENTS"]
data_dir = "/home/jovyan/work/data/"

use_header = True
infer_schema = True
single_file = True
repartition_num = 100

In [4]:

for _ in (0, 1):
    parquet_path = convert_csv_to_parquet_spark(
        spark=spark,
        input_csv=input_csv[_],
        output_parquet=output_parquet[_],
        header=use_header,
        infer_schema=infer_schema,
        repartition=repartition_num,
        single_file=single_file,
        data_dir=data_dir,
    )

📥 CSV de entrada : /home/jovyan/work/data/BEHAVIOURAL.csv
📤 Parquet salida: /home/jovyan/work/data/BEHAVIOURAL
🔹 DataFrame leído con 1724854 filas y 14 columnas
🔹 Reparticionando a 100 particiones
🔹 Uniendo la salida en un único archivo Parquet con coalesce(1)
✅ Conversión a Parquet terminada.
📥 CSV de entrada : /home/jovyan/work/data/CLIENTS.csv
📤 Parquet salida: /home/jovyan/work/data/CLIENTS
🔹 DataFrame leído con 162977 filas y 45 columnas
🔹 Reparticionando a 100 particiones
🔹 Uniendo la salida en un único archivo Parquet con coalesce(1)
✅ Conversión a Parquet terminada.


In [5]:
df_parquet = spark.read.parquet(parquet_path)
df_csv = spark.read.csv("/home/jovyan/work/data/BEHAVIOURAL.csv", header=True)
df_csv.show(5)
df_parquet.show(5)
df_parquet.printSchema()

+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|       CONTRACT_ID|   CLIENT_ID|      DATE|CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|CURRENCY|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|ES1821000018d00XXX|ES182394447V|2021-08-29|              491.21|            540.0|                     0.0|               28.03|                   28.03|                       0.0|              46.81| 

In [6]:
parquet_path

'/home/jovyan/work/data/CLIENTS'